In [1]:
from pathlib import Path
from typing import Any, Dict
import numpy as np
import os
import numpy.typing as npt
import pandas as pd
from lmfit import Parameters
from echopop import inversion
from echopop.nwfsc_feat import (
    apportion,
    biology, 
    FEAT,
    ingest_nasc, 
    get_proportions, 
    kriging,
    load_data, 
    spatial,
    stratified,
    transect, 
    utils,
    variogram
)

c:\ProgramData\anaconda3\envs\get_echopop_running\Lib\site-packages\pandera\_pandas_deprecated.py:160: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


In [22]:
Year=2019 
runyearstr=str(Year) #added by RT
runyearstr
NASC_year_path="Exports/"+runyearstr+"/"  
Strata_year_path="Stratification/"+runyearstr+"/"  

In [3]:
DATA_ROOT = Path("C:/rthomas/Projects/EchoPop_validation/Data")
# DATA_ROOT = Path("C:/Users/Brandyn Lucca/Documents/Data/echopop_2019")


# ==================================================================================================
# ==================================================================================================
# DATA INGESTION
# ==================================================================================================
# Organize NASC file
# ------------------

# Merge exports
df_intervals, df_exports = ingest_nasc.merge_echoview_nasc(
    #nasc_path = DATA_ROOT / "raw_nasc/",
    #nasc_path = DATA_ROOT / "Exports/2019/",
    #nasc_path = DATA_ROOT / "Exports/"+runyearstr+"/",  #doesn't work if you put these together like this
    nasc_path = DATA_ROOT / NASC_year_path,
    filename_transect_pattern = r"T(\d+)",
    default_transect_spacing = 10.0,
    default_latitude_threshold = 60.0,
)


In [4]:
#df_intervals
df_exports
#seems like it is loading nasc data okay

,region_id,region_name,region_class,process_id,interval,layer,sv_mean,nasc,standard_deviation,transect_num,distance_s,distance_e,ping_date,ping_time,latitude,longitude,max_depth,transect_spacing,layer_depth_min,layer_depth_max
0,3,Herring-Region 3,herring,149201,454,7,-999.000000,0.000000,0.000000e+00,87.0,226.50210,226.999529,20190819,19:55:23.2380,48.731117,-125.368693,104.566585,14.977436,60.0,70.0
1,3,Herring-Region 3,herring,149201,454,8,-86.098209,0.135606,3.208852e-08,87.0,226.50210,226.999529,20190819,19:55:23.2380,48.731117,-125.368693,104.566585,14.977436,70.0,80.0
2,3,Herring-Region 3,herring,149201,454,9,-84.858697,0.180397,6.425318e-08,87.0,226.50210,226.999529,20190819,19:55:23.2380,48.731117,-125.368693,104.566585,14.977436,80.0,90.0
3,3,Herring-Region 3,herring,149201,454,10,-50.938691,444.867027,3.353771e-05,87.0,226.50210,226.999529,20190819,19:55:23.2380,48.731117,-125.368693,104.566585,14.977436,90.0,100.0
4,3,Herring-Region 3,herring,149201,454,11,-68.915509,3.942757,7.320723e-07,87.0,226.50210,226.999529,20190819,19:55:23.2380,48.731117,-125.368693,104.566585,14.977436,100.0,110.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136163,40,Region 40,mesopelagics,148597,19627,21,-86.704106,0.555822,2.032088e-08,86.0,9813.00249,9813.431812,20190818,05:05:16.0170,48.564564,-126.715015,759.765513,10.007759,200.0,210.0
136164,40,Region 40,mesopelagics,148597,19627,22,-81.626612,1.706778,3.531383e-08,86.0,9813.00249,9813.431812,20190818,05:05:16.0170,48.564564,-126.715015,759.765513,10.007759,210.0,220.0
136165,40,Region 40,mesopelagics,148597,19627,23,-82.433394,1.319372,3.040927e-08,86.0,9813.00249,9813.431812,20190818,05:05:16.0170,48.564564,-126.715015,759.765513,10.007759,220.0,230.0
136166,40,Region 40,mesopelagics,148597,19627,24,-81.467710,1.311439,3.522668e-08,86.0,9813.00249,9813.431812,20190818,05:05:16.0170,48.564564,-126.715015,759.765513,10.007759,230.0,240.0


In [5]:
#!pip install -e c:/rthomas/pycode/echopop/
#would do this if wanted to use a kernel that didn't have EchoPop in it (e.g. one way to circumvent whatever bug is going on in vscode.  
#Once install echopop, would stay in the kernel)
#Note: this is something to do only from a notebook (with the ! operator)

In [ ]:

# ==================================================================================================
# Read in transect-region-haul keys
# ---------------------------------
TRANSECT_REGION_FILEPATH_ALL_AGES = (
    DATA_ROOT / "Stratification/2019/US&CAN_2019_transect_region_haul_age1+ auto_final.xlsx"
)
TRANSECT_REGION_FILEPATH_NO_AGE1 = (
    DATA_ROOT / "Stratification/2019/US&CAN_2019_transect_region_haul_age2+ auto_20191205.xlsx"
)
TRANSECT_REGION_FILE_RENAME: dict = {
    "tranect": "transect_num",
    "region id": "region_id",
    "trawl #": "haul_num",
}
TRANSECT_REGION_SHEETNAME_ALL_AGES: str = "Sheet1"
TRANSECT_REGION_SHEETNAME_NO_AGE1: str = "Sheet1"

# Read in the transect-region-haul key files for each group
transect_region_haul_key_all_ages: pd.DataFrame = ingest_nasc.read_transect_region_haul_key(
    filename=TRANSECT_REGION_FILEPATH_ALL_AGES,
    sheetname=TRANSECT_REGION_SHEETNAME_ALL_AGES,
    rename_dict=TRANSECT_REGION_FILE_RENAME,
)

transect_region_haul_key_no_age1: pd.DataFrame = ingest_nasc.read_transect_region_haul_key(
    TRANSECT_REGION_FILEPATH_NO_AGE1, TRANSECT_REGION_SHEETNAME_NO_AGE1, TRANSECT_REGION_FILE_RENAME
)


C:\rthomas\Projects\EchoPop_validation\Data\Stratification\2019\US&CAN_2019_transect_region_haul_age1+ auto_final.xlsx


In [9]:
# ==================================================================================================
# Read in transect-region-haul keys
# ---------------------------------
REGION_NAME_EXPR_DICT: Dict[str, dict] = {
    "REGION_CLASS": {
        "Age-1 Hake": "^(?:h1a(?![a-z]|m))",
        "Age-1 Hake Mix": "^(?:h1am(?![a-z]|1a))",
        "Hake": "^(?:h(?![a-z]|1a)|hake(?![_]))",
        "Hake Mix": "^(?:hm(?![a-z]|1a)|hake_mix(?![_]))",
    },
    "HAUL_NUM": {
        "[0-9]+",
    },
    "COUNTRY": {
        "CAN": "^[cC]",
        "US": "^[uU]",
    },
}

# Process the region name codes to define the region classes
# e.g. H5C - Region 2 corresponds to "Hake, Haul #5, Canada"
df_exports_with_regions: pd.DataFrame = ingest_nasc.process_region_names(
    df=df_exports,
    region_name_expr_dict=REGION_NAME_EXPR_DICT,
    can_haul_offset=200,
)


In [10]:
# ==================================================================================================
# [OPTIONAL] Generate transect-region-haul key from compiled values
# ---------------------------------

# Generate transect-region-haul key from compiled values
df_transect_region_haul_key_no_age1: pd.DataFrame = ingest_nasc.generate_transect_region_haul_key(
    df=df_exports_with_regions, 
    filter_list=["Hake", "Hake Mix"]
)

df_transect_region_haul_key_all_ages = ingest_nasc.generate_transect_region_haul_key(
    df=df_exports_with_regions, 
    filter_list=["Age-1 Hake", "Age-1", "Hake", "Hake Mix"]
)



In [13]:
# ==================================================================================================
# Consolidate the Echvoiew NASC export files
# ------------------------------------------
df_nasc_no_age1: pd.DataFrame = ingest_nasc.consolidate_echvoiew_nasc(
    df_merged=df_exports_with_regions,
    interval_df=df_intervals,
    region_class_names=["Hake", "Hake Mix"],
    impute_region_ids=True,
    transect_region_haul_key_df=transect_region_haul_key_no_age1,
)

df_nasc_all_ages: pd.DataFrame = ingest_nasc.consolidate_echvoiew_nasc(
    df_merged=df_exports_with_regions,
    interval_df=df_intervals,
    region_class_names=["Age-1 Hake", "Age-1", "Hake", "Hake Mix"],
    impute_region_ids=True,
    transect_region_haul_key_df=transect_region_haul_key_all_ages,
)
df_nasc_all_ages


,transect_num,region_id,distance_s,distance_e,latitude,longitude,transect_spacing,layer_mean_depth,layer_height,bottom_depth,nasc,haul_num
645,1.0,999,744.016009,744.491145,34.397267,-121.143005,10.0,0.0,0.0,0.0,0.0,0.0
647,1.0,999,744.500276,744.995605,34.397391,-121.133196,10.0,0.0,0.0,0.0,0.0,0.0
649,1.0,999,745.004125,745.499447,34.397435,-121.123057,10.0,0.0,0.0,0.0,0.0,0.0
651,1.0,999,745.508199,745.994306,34.397394,-121.112871,10.0,0.0,0.0,0.0,0.0,0.0
653,1.0,999,746.003214,746.495701,34.397437,-121.102888,10.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
4425,145.0,999,3073.001555,3073.499465,51.897250,-131.025358,10.0,0.0,0.0,0.0,0.0,0.0
4427,145.0,999,3073.501846,3073.998088,51.897212,-131.011859,10.0,0.0,0.0,0.0,0.0,0.0
4429,145.0,999,3074.000343,3074.498887,51.896882,-130.998452,10.0,0.0,0.0,0.0,0.0,0.0
4431,145.0,999,3074.501062,3074.998484,51.897065,-130.984977,10.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# # ==================================================================================================
# # [OPTIONAL] Read in a pre-consolidated NASC data file
# # ----------------------------------------------------
# FEAT_TO_ECHOPOP_COLUMNS: Dict[str, str] = {
#     "transect": "transect_num",
#     "region id": "region_id",
#     "vessel_log_start": "distance_s",
#     "vessel_log_end": "distance_e",
#     "spacing": "transect_spacing",
#     "layer mean depth": "layer_mean_depth",
#     "layer height": "layer_height",
#     "bottom depth": "bottom_depth",
#     "assigned haul": "haul_num",
# }

# #
# df_nasc_all_ages: pd.DataFrame = ingest_nasc.read_nasc_file(
#     filename=DATA_ROOT / "Exports/US_CAN_NASC_2019_table_all_ages.xlsx",
#     sheetname="Sheet1",
#     column_name_map=FEAT_TO_ECHOPOP_COLUMNS,
# )


In [ ]:

# ==================================================================================================
# [OPTIONAL] Convert the NASC DataFrame format from AFSC to FEAT
# --------------------------------------------------------------

# EXAMPLE: 2001 Dataset
# df_nasc_all_ages_feat = ingest_nasc.convert_afsc_nasc_to_feat(
#     df=df_nasc_all_ages,
#     default_interval_distance = 0.5,
#     default_transect_spacing = 10.0,
#     inclusion_filter = {"transect_num", np.arange(1, 200)},
# )


TypeError: unhashable type: 'numpy.ndarray'

In [ ]:
# ==================================================================================================
# [OPTIONAL] Filter the transect intervals to account for on- and off-effort
# --------------------------------------------------------------------------

# DataFrame with filtered intervals representing on-effort
# df_nasc_all_ages_cleaned: pd.DataFrame = ingest_nasc.filter_transect_intervals(
#     nasc_df=df_nasc_all_ages_feat,
#     transect_filter_df=Path("Path/to/file"),
#     subset_filter="survey == 201003",
#     transect_filter_sheet="Sheet1",
# )


NameError: name 'df_nasc_all_ages_feat' is not defined

In [ ]:
# ==================================================================================================
# Load in the biolodical data
# ---------------------------
BIODATA_SHEET_MAP: Dict[str, str] = {
    "catch": "biodata_catch", 
    "length": "biodata_length",
    "specimen": "biodata_specimen",
}
SUBSET_DICT: Dict[Any, Any] = {
    "ships": {
        160: {
            "survey": 201906
        },
        584: {
            "survey": 2019097,
            "haul_offset": 200
        }
    },
    "species_code": [22500]
}
FEAT_TO_ECHOPOP_BIODATA_COLUMNS = {
    "frequency": "length_count",
    "haul": "haul_num",
    "weight_in_haul": "weight",
}
BIODATA_LABEL_MAP: Dict[Any, Dict] = {
    "sex": {
        1: "male",
        2: "female",
        3: "unsexed"
    }
}

# 
dict_df_bio = load_data.load_biological_data(
    biodata_filepath=DATA_ROOT / "Biological/Updated/1977-2023_Survey_Biodata_old_not_updated.xlsx", 
    biodata_sheet_map=BIODATA_SHEET_MAP, 
    column_name_map=FEAT_TO_ECHOPOP_BIODATA_COLUMNS, 
    subset_dict=SUBSET_DICT, 
    biodata_label_map=BIODATA_LABEL_MAP
)

{'catch':        ship   survey  haul_num species_code  \
 11033   160   201906       1.0        22500   
 11045   160   201906       3.0        22500   
 11054   160   201906       7.0        22500   
 11061   160   201906       8.0        22500   
 11069   160   201906       9.0        22500   
 ...     ...      ...       ...          ...   
 12805   584  2019097     213.0        22500   
 12818   584  2019097     214.0        22500   
 12830   584  2019097     216.0        22500   
 12850   584  2019097     218.0        22500   
 12883   584  2019097     222.0        22500   
 
                               species_name  number_in_haul  weight  \
 11033  Pacific hake - Merluccius productus           769.0   56.25   
 11045  Pacific hake - Merluccius productus           866.0   46.25   
 11054  Pacific hake - Merluccius productus           848.0   51.90   
 11061  Pacific hake - Merluccius productus          1415.0  103.10   
 11069  Pacific hake - Merluccius productus           173.

In [27]:
# ==================================================================================================
# Load in strata files
# --------------------
STRATA_SHEET_MAP = {
    "inpfc": "INPFC",
    "ks": "Base KS",
}
FEAT_TO_ECHOPOP_STRATA_COLUMNS = {
    "fraction_hake": "nasc_proportion",
    "haul": "haul_num",
    "stratum": "stratum_num",
}

#
df_dict_strata = load_data.load_strata(
    #strata_filepath=DATA_ROOT / "Stratification/US_CAN strata 2019_final.xlsx", 
    strata_filepath=DATA_ROOT / Strata_year_path / "Stratification_geographic_Lat_2019_EP_08-Jan-2025.xlsx", 
    strata_sheet_map=STRATA_SHEET_MAP, 
    column_name_map=FEAT_TO_ECHOPOP_STRATA_COLUMNS
)


In [42]:
# ==================================================================================================
# Load in geographical strata files
# ---------------------------------
GEOSTRATA_SHEET_MAP = {
    "inpfc": "INPFC",
    "ks": "Base KS",
}
FEAT_TO_ECHOPOP_GEOSTRATA_COLUMNS = {
    "northlimit_latitude": "northlimit_latitude",
    "stratum": "stratum_num",
}

# 
df_dict_geostrata = load_data.load_geostrata(
    #geostrata_filepath=DATA_ROOT / Strata_year_path / "Stratification_geographic_Lat_2019_final.xlsx", 
    geostrata_filepath=DATA_ROOT / Strata_year_path / "Stratification_geographic_Lat_2019_EP_08-Jan-2025.xlsx", 
    geostrata_sheet_map=GEOSTRATA_SHEET_MAP, 
    column_name_map=FEAT_TO_ECHOPOP_GEOSTRATA_COLUMNS
)


In [28]:
# ==================================================================================================
# Stratify data based on haul numbers
# -----------------------------------

# Add INPFC
# ---- NASC
df_nasc_all_ages = load_data.join_strata_by_haul(data=df_nasc_all_ages, 
                                                 strata_df=df_dict_strata["inpfc"],
                                                 stratum_name="stratum_inpfc") 
# ---- Biodata
dict_df_bio = load_data.join_strata_by_haul(dict_df_bio,
                                            df_dict_strata["inpfc"],
                                            stratum_name="stratum_inpfc")

# Add KS
# ---- NASC
df_nasc_all_ages = load_data.join_strata_by_haul(df_nasc_all_ages, 
                                                 df_dict_strata["ks"],
                                                 stratum_name="stratum_ks") 
df_nasc_no_age1 = load_data.join_strata_by_haul(df_nasc_no_age1, 
                                                df_dict_strata["ks"],
                                                stratum_name="stratum_ks") 
# ---- Biodata
dict_df_bio = load_data.join_strata_by_haul(dict_df_bio,
                                            df_dict_strata["ks"],
                                            stratum_name="stratum_ks") 


In [32]:
# ==================================================================================================
# Load kriging mesh file
# ----------------------

FEAT_TO_ECHOPOP_MESH_COLUMNS = {
    "centroid_latitude": "latitude",
    "centroid_longitude": "longitude",
    "fraction_cell_in_polygon": "fraction",
}

# 
df_mesh = load_data.load_mesh_data(
    mesh_filepath=DATA_ROOT / "Kriging files & parameters/Kriging grid files/krig_grid2_5nm_cut_centroids_2013.xlsx", 
    sheet_name="krigedgrid2_5nm_forChu", 
    column_name_map=FEAT_TO_ECHOPOP_MESH_COLUMNS
)


In [43]:

# ==================================================================================================
# [OPTIONAL] Stratify data based on latitude intervals
# ----------------------------------------------------
# INPFC (from geostrata)
df_nasc_all_ages = load_data.join_geostrata_by_latitude(df_nasc_all_ages, 
                                                        df_dict_geostrata["inpfc"],
                                                        stratum_name="geostratum_inpfc")
df_nasc_no_age1 = load_data.join_geostrata_by_latitude(df_nasc_no_age1, 
                                                       df_dict_geostrata["inpfc"],
                                                       stratum_name="geostratum_inpfc")
# KS (from geostrata)
df_nasc_all_ages = load_data.join_geostrata_by_latitude(df_nasc_all_ages, 
                                                        df_dict_geostrata["ks"],
                                                        stratum_name="geostratum_ks")
df_nasc_no_age1 = load_data.join_geostrata_by_latitude(df_nasc_no_age1, 
                                                       df_dict_geostrata["ks"],
                                                       stratum_name="geostratum_ks")

# MESH
# ---- DataFrame merged with geographically distributed stratum number (KS or INPFC)
# -------- INPFC (from geostrata)
df_mesh = load_data.join_geostrata_by_latitude(df_mesh, 
                                               df_dict_geostrata["inpfc"], 
                                               stratum_name="geostratum_inpfc")
# -------- KS (from geostrata)
df_mesh = load_data.join_geostrata_by_latitude(df_mesh, 
                                               df_dict_geostrata["ks"], 
                                               stratum_name="geostratum_ks")


In [45]:
# ==================================================================================================
# Load kriging and variogram parameters
# -------------------------------------

FEAT_TO_ECHOPOP_GEOSTATS_PARAMS_COLUMNS = {
    "hole": "hole_effect_range",
    "lscl": "correlation_range",
    "nugt": "nugget",
    "powr": "decay_power",
    "ratio": "aspect_ratio",
    "res": "lag_resolution",
    "srad": "search_radius",
}

# 
dict_kriging_params, dict_variogram_params = load_data.load_kriging_variogram_params(
    geostatistic_params_filepath=(
        DATA_ROOT / "Kriging files & parameters/2019/default_vario_krig_settings_2019_US&CAN.xlsx"
    ),
    sheet_name="Sheet1",
    column_name_map=FEAT_TO_ECHOPOP_GEOSTATS_PARAMS_COLUMNS
)


In [49]:
# ==================================================================================================
# ==================================================================================================
# DATA PROCESSING
# ==================================================================================================
# Generate binned distributions [age, length]
# -------------------------------------------
AGE_BINS: npt.NDArray[np.number] = np.linspace(start=1., stop=22., num=22)
LENGTH_BINS: npt.NDArray[np.number] = np.linspace(start=2., stop=80., num=40)

# 
# ---- Length
utils.binify(
    data=dict_df_bio, bins=LENGTH_BINS, bin_column="length", 
)

# Age
utils.binify(
    data=dict_df_bio, bins=AGE_BINS, bin_column="age",
)


In [ ]:
dict_df_bio["specimen"]

,ship,survey,haul_num,species_code,sex,length,weight,specimen_number,age,maturity_code,...,age_structure,maturity_table,age_class,merged_species_code,ex_from_akey,ex_from_lkey,modfied_by,modified_date,length_bin,age_bin
70873,160,201906,1.0,22500,all,24.0,0.08,100620381.0,1.0,1,...,1,1.0,NaN,22500,N,N,abillings,2019-10-16 00:00:00,"(23.0, 25.0]","(0.5, 1.5]"
70874,160,201906,1.0,22500,all,23.0,0.06,100620382.0,1.0,1,...,1,1.0,NaN,22500,N,N,abillings,2019-10-16 00:00:00,"(21.0, 23.0]","(0.5, 1.5]"
70875,160,201906,1.0,22500,all,22.0,0.06,100620383.0,1.0,1,...,1,1.0,NaN,22500,N,N,abillings,2019-10-16 00:00:00,"(21.0, 23.0]","(0.5, 1.5]"
70876,160,201906,1.0,22500,all,22.0,0.06,100620384.0,1.0,1,...,1,1.0,NaN,22500,N,N,abillings,2019-10-16 00:00:00,"(21.0, 23.0]","(0.5, 1.5]"
70877,160,201906,1.0,22500,all,23.0,0.06,100620385.0,1.0,1,...,1,1.0,NaN,22500,N,N,abillings,2019-10-16 00:00:00,"(21.0, 23.0]","(0.5, 1.5]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78968,584,2019097,222.0,22500,all,46.0,0.631,15996677.0,9.0,2,...,1,1.0,NaN,22500,NaN,NaN,abillings,2019-11-19 00:00:00,"(45.0, 47.0]","(8.5, 9.5]"
78969,584,2019097,222.0,22500,all,46.0,0.659,15996678.0,6.0,2,...,1,1.0,NaN,22500,NaN,NaN,abillings,2019-11-19 00:00:00,"(45.0, 47.0]","(5.5, 6.5]"
78970,584,2019097,222.0,22500,all,52.0,0.911,15996679.0,9.0,2,...,1,1.0,NaN,22500,NaN,NaN,abillings,2019-11-19 00:00:00,"(51.0, 53.0]","(8.5, 9.5]"
78971,584,2019097,222.0,22500,all,48.0,0.832,15996680.0,9.0,2,...,1,1.0,NaN,22500,NaN,NaN,abillings,2019-11-19 00:00:00,"(47.0, 49.0]","(8.5, 9.5]"


In [ ]:
# ==================================================================================================
# Fit length-weight regression to the binned data
# -----------------------------------------------

# Dictionary for length-weight regression coefficients
dict_length_weight_coefs = {}

# For all fish
dict_length_weight_coefs["all"] = dict_df_bio["specimen"].assign(sex="all").groupby(["sex"]).apply(
    biology.fit_length_weight_regression,
    include_groups=False
)

# Sex-specific
dict_length_weight_coefs["sex"] = dict_df_bio["specimen"].groupby(["sex"]).apply(
    biology.fit_length_weight_regression,
    include_groups=False
)


TypeError: loop of ufunc does not support argument 0 of type float which has no callable log10 method